In [10]:

import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin


class RidgeReg(RegressorMixin, BaseEstimator):
    def __init__(self, learning_rate=0.01, epochs=1000, alpha=0.01):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha

        if alpha < 0:
            raise ValueError("alpha can't be negative")

    def fit(self, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        if x.ndim != 2:
            raise ValueError("X must be a 2D array.")
        if len(x) != len(y) or len(y) == 0:
            raise ValueError("X and y must have equal, nonzero lengths.")
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError("X and y must contain only finite values.")

        self.n_features_in_ = x.shape[1]
        n_samples, n_features = x.shape

        self.w = np.zeros(n_features)
        self.b = 0.0
        self.loss_history = []

        for epoch in range(self.epochs):
            y_pred = x @ self.w + self.b
            error = y_pred - y

            dw = (2 / n_samples) * (
                x.T @ error + self.alpha * self.w
            )
            db = 2 * np.mean(error)

            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            updated_pred = x @ self.w + self.b
            mse = np.mean((updated_pred - y) ** 2)
            penalty = (
                self.alpha / n_samples
            ) * np.sum(self.w ** 2)

            self.loss_history.append(mse + penalty)

        # Standard fitted-attribute names
        self.coef_ = self.w.copy()
        self.intercept_ = self.b

        return self

    def predict(self, X):
        if not hasattr(self, "w"):
            raise ValueError("Fit the model before predicting.")

        X = np.asarray(X, dtype=float)

        if X.ndim != 2:
            raise ValueError("X must be a 2D array.")
        if X.shape[1] != self.n_features_in_:
            raise ValueError("X has an incorrect number of features.")

        return X @ self.w + self.b


In [11]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge,LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score

data=load_diabetes()
x=data.data
y=data.target

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

models = {
    "Scratch Ridge": make_pipeline(
        StandardScaler(),
        RidgeReg(learning_rate=0.01, epochs=1000, alpha=1.0)
    ),
    "Sklearn Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),
    "Linear Regression": make_pipeline(
        StandardScaler(),
        LinearRegression()
    )
}

# Train every pipeline
for name, model in models.items():
    model.fit(x_train, y_train)

# Evaluate every pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

for name, model in models.items():
    predictions = model.predict(x_test)

    print(f"\n{name}")
    print("MAE:", mean_absolute_error(y_test, predictions))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
    print("R²:", r2_score(y_test, predictions))



Scratch Ridge
MAE: 42.88068378630262
RMSE: 53.70749286068893
R²: 0.4555658413920354

Sklearn Ridge
MAE: 42.81199941834888
RMSE: 53.77745406534389
R²: 0.45414652070698236

Linear Regression
MAE: 42.79409467959994
RMSE: 53.85344583676593
R²: 0.4526027629719195
